# Tutorial 6: Naive Bayes
## Combine prior probabilities and feature evidence while questioning the “naive” assumptions

**Course:** IE 1171  
**File used:** `processed.cleveland.csv`  
**Level 1:** Required core—Gaussian Naive Bayes  
**Level 2:** Optional deep dives—feature types, independence, and smoothing

---


Claude will draft the implementation. Your responsibility is to verify how the target is coded, distinguish continuous and discrete predictors, connect the program to Bayes’ theorem, inspect false positives and false negatives, and avoid treating an instructional model as a clinical decision system.


# <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="44" style="vertical-align:middle; margin-right:10px;"> Learning Objectives

By the end of this tutorial, you should be able to:

1. **Understand probability-based classification**
   - Identify the prior, likelihood, evidence term, and posterior in Bayes’ theorem.
   - Explain the conditional-independence assumption behind the word “naive.”
   - Distinguish Gaussian, Bernoulli, and multinomial Naive Bayes.
2. **Build and evaluate a heart-disease classifier**
   - Distinguish continuous measurements from coded categories in the supplied data.
   - Fit a Gaussian Naive Bayes model using fixed model-fitting and evaluation rows.
   - Interpret posterior probabilities, a confusion matrix, and complementary performance measures.
3. **Judge health predictions responsibly**
   - Explain why false positives and false negatives have different consequences.
   - Compare model forms, assumptions, smoothing choices, and group error patterns.
   - Describe why this educational prediction is not a diagnosis.


# <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="44" style="vertical-align:middle; margin-right:10px;"> Assigned Reading

## Suggested Reading

**James et al., _An Introduction to Statistical Learning_**

- Section 4.4.4: Naive Bayes

Focus on:

- Bayes’ theorem;
- class priors;
- class-conditional feature distributions;
- the conditional-independence assumption;
- posterior class probabilities;
- classification by the largest posterior probability.


# Dataset and Variable Roles

The supplied file is:

```text
processed.cleveland.csv
```

The expected response is:

- `heartdisease`: `0` = no presence of heart disease; `1` = presence of heart disease.

Expected predictors from the original notebook:

| Type | Variables |
|---|---|
| Continuous | `age`, `restbps`, `cholesterol`, `thalach`, `oldpeak` |
| Binary or categorical codes | `sex`, `chestpain`, `fastingbs`, `restecg`, `exerciseang`, `slope`, `ca`, `thal` |

The numeric codes for categorical variables are labels, not measured quantities. A larger code does not automatically mean “more” of the underlying concept.


# <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="44" style="vertical-align:middle; margin-right:10px;"> Tutorial Flow

| Section | Purpose |
|---|---|
| **Theory** | Translate Bayes’ theorem and model assumptions into code. |
| **Manual Pause** | Reason about priors, evidence, and errors before coding. |
| **Claude Coding Task** | Ask for one focused, testable program. |
| **Your Workspace** | Read and run Claude’s output. |
| **Reference Solution** | Compare after attempting the task. |
| **Human Check** | Verify variables, distributions, probabilities, and claims. |
| **Look Back** | Decide what the model can and cannot support. |

> A probabilistic output is not automatically a trustworthy probability. It depends on the data, feature representation, assumptions, and intended use.


## Tutorial Symbols

| Symbol | Meaning | What to do |
|---|---|---|
| <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="28" style="vertical-align:middle; margin-right:8px;"> | **Tutorial Flow** | Follow the notebook’s normal route. |
| <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="28" style="vertical-align:middle; margin-right:8px;"> | **Learning Objectives** | See the three destinations for the tutorial. |
| <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="28" style="vertical-align:middle; margin-right:8px;"> | **Assigned Reading** | Read the named sections before or alongside the notebook. |
| <img src="tutorial-icons/theory.png" alt="Theory" width="28" style="vertical-align:middle; margin-right:8px;"> | **Theory** | Connect the assigned reading to the current part. |
| <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="28" style="vertical-align:middle; margin-right:8px;"> | **Manual Pause** | Think or predict before asking Claude. |
| <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="28" style="vertical-align:middle; margin-right:8px;"> | **Without Claude** | Notice the programming details Claude can coordinate. |
| <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="28" style="vertical-align:middle; margin-right:8px;"> | **Claude Task** | Use one focused and checkable prompt. |
| <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="28" style="vertical-align:middle; margin-right:8px;"> | **Your Workspace** | Paste, read, and run Claude’s response. |
| <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="28" style="vertical-align:middle; margin-right:8px;"> | **Reference Solution** | Compare only after your own attempt. |
| <img src="tutorial-icons/human_check.png" alt="Human Check" width="28" style="vertical-align:middle; margin-right:8px;"> | **Human Check** | Verify the data, code, output, and claim yourself. |
| <img src="tutorial-icons/look_back.png" alt="Look Back" width="28" style="vertical-align:middle; margin-right:8px;"> | **Look Back** | Interpret, challenge, and reflect on the result. |
| <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="28" style="vertical-align:middle; margin-right:8px;"> | **Level 2 — Challenge Ahead** | Take the optional harder route after Level 1. |


# <img src="tutorial-icons/theory.png" alt="Theory" width="44" style="vertical-align:middle; margin-right:10px;"> Theory Foundation: Why Use Naive Bayes—and Why Dimension Changes the Problem

Naive Bayes is useful when we need a fast probabilistic baseline, have limited training data, or work with many features such as words, symptoms, or binary indicators. Bayes' rule gives

$$
P(C=k\mid x)=\frac{P(C=k)P(x\mid C=k)}{\sum_{\ell}P(C=\ell)P(x\mid C=\ell)}.
$$

A fully general class-conditional joint distribution $P(X_1,\ldots,X_p\mid C)$ is extremely difficult to estimate. If every one of $p$ categorical features has $m$ levels, the joint table has $m^p$ possible cells per class. With binary features, that is $2^p$: 20 features already create 1,048,576 possible patterns. Most patterns will be unseen unless $n$ is enormous.

Naive Bayes replaces that joint model with conditional independence:

$$
P(x\mid C=k)=\prod_{j=1}^{p}P(x_j\mid C=k).
$$

This reduces the number of feature-distribution parameters from exponential in $p$ to roughly linear in $p$. That is the main reason Naive Bayes can work surprisingly well in high-dimensional, sparse settings. It is not because high dimension becomes harmless; it is because a strong assumption makes estimation tractable.

Dimension still causes trouble. Data occupy an exponentially growing space, observed cases become far apart, density estimates become sparse, and irrelevant features can add noisy evidence. If correlated features repeat the same signal, the product can count that evidence multiple times and produce overconfident probabilities. Computation is stabilized with log scores,

$$
\log \tilde P(C=k\mid x)=\log P(C=k)+\sum_{j=1}^{p}\log P(x_j\mid C=k),
$$

then scores are normalized with a log-sum-exp calculation. Gaussian NB models continuous features by class-specific means and variances; Bernoulli NB models binary indicators; multinomial NB models nonnegative counts. Smoothing prevents unseen events from forcing an entire product to zero.

Use Naive Bayes when its speed, small-sample efficiency, interpretability as accumulated evidence, and baseline value fit the task. Do not use its posterior as unquestioned truth: inspect feature types, dependence, distributional fit, calibration, class imbalance, and error consequences.

### Questions you should be ready to answer

- What makes a general joint distribution exponential in dimension?
- Which assumption changes exponential parameter growth to roughly linear growth?
- Why can classification remain useful while probabilities are poorly calibrated?
- When should Gaussian, Bernoulli, or multinomial Naive Bayes be used?

# Level 1 — Required Core

# Part 1: Bayes’ Theorem and the Classification Question

Bayes’ theorem is:

$$
P(C\mid X)=\frac{P(X\mid C)P(C)}{P(X)}
$$

For this tutorial:

- $C$ is the heart-disease class;
- $P(C)$ is the class prior;
- $P(X\mid C)$ is the likelihood of the observed predictors under that class;
- $P(C\mid X)$ is the posterior probability after observing the predictors.

Naive Bayes assumes that predictors are conditionally independent given the class:

$$
P(X_1,\ldots,X_p\mid C)
=
\prod_{j=1}^{p}P(X_j\mid C)
$$

That assumption makes the calculation simple. It can also be unrealistic.

## Required Level 1 Choice

Level 1 begins with **Gaussian Naive Bayes using only the five continuous predictors**. This keeps the feature distribution assumption visible instead of silently treating category codes as continuous Gaussian measurements.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: From Prior Belief to Posterior Group Probability

Bayes’ theorem combines how common a class is before seeing the predictors (the prior) with how compatible the observed predictors are with that class (the likelihood). The result is a posterior probability for each outcome group. Classification chooses the group with the largest posterior only after those probabilities are calculated.

### Deeper explanation

For two classes, compare unnormalized scores $P(C=1)P(x\mid C=1)$ and $P(C=0)P(x\mid C=0)$; the common evidence denominator $P(x)$ is unnecessary for the winning class but necessary for normalized probabilities. Priors matter most when likelihood evidence is weak. Changing prevalence at deployment can therefore change posterior probabilities even if class-conditional feature patterns stay fixed. This is one reason probability calibration must be checked in the intended population.


## <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="36" style="vertical-align:middle; margin-right:9px;"> Manual Pause: Think Before Coding

Answer briefly:

1. What is the response variable?
2. What do classes `0` and `1` mean?
3. What is the class prior?
4. Which five predictors are treated as continuous?
5. Why should a histogram be inspected before assuming a Gaussian shape?
6. Why is correlation between predictors relevant to the naive assumption?
7. What is a false negative in this application?
8. What is a false positive?
9. Which error might be especially concerning in a screening context?
10. Why would that answer depend on the real use, follow-up process, and costs?


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 1: Load and Audit the Heart-Disease File

```text
Act as a careful Python tutor.

Write one Jupyter Notebook code cell that:

1. imports pathlib and pandas;
2. loads processed.cleveland.csv into a dataframe named heart;
3. confirms that the expected 13 predictors and heartdisease are present;
4. prints the shape, first five rows, missing-value counts, and data types;
5. prints heartdisease class counts and proportions;
6. verifies that heartdisease contains only 0 and 1;
7. creates two lists named continuous_features and discrete_features using
   the variable roles supplied in the tutorial;
8. does not drop data and does not fit a model.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 1


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 1


In [ ]:
from pathlib import Path
import pandas as pd

heart_path = Path("processed.cleveland.csv")
assert heart_path.exists(), f"File not found: {heart_path.resolve()}"

continuous_features = [
    "age",
    "restbps",
    "cholesterol",
    "thalach",
    "oldpeak",
]

discrete_features = [
    "sex",
    "chestpain",
    "fastingbs",
    "restecg",
    "exerciseang",
    "slope",
    "ca",
    "thal",
]

target_column = "heartdisease"
expected_columns = continuous_features + discrete_features + [target_column]

heart = pd.read_csv(heart_path)

missing_columns = sorted(set(expected_columns) - set(heart.columns))
assert not missing_columns, f"Missing required columns: {missing_columns}"

heart = heart[expected_columns].copy()

print("Shape:", heart.shape)
display(heart.head())

print("\nMissing values:")
display(heart.isna().sum().to_frame("missing_count"))

print("\nData types:")
display(heart.dtypes.to_frame("dtype"))

print("\nClass counts:")
display(heart[target_column].value_counts(dropna=False).sort_index().to_frame("count"))

print("\nClass proportions:")
display(
    heart[target_column]
    .value_counts(normalize=True, dropna=False)
    .sort_index()
    .rename("proportion")
    .to_frame()
)

observed_classes = set(heart[target_column].dropna().unique())
assert observed_classes.issubset({0, 1}), (
    f"Unexpected heartdisease values: {sorted(observed_classes)}"
)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

- Are all expected variables present?
- Are missing values already resolved in the processed file?
- Are the category codes documented?
- Is the target binary?
- Is one class substantially more common?
- Did Claude preserve the raw dataframe?
- Did Claude avoid treating a code such as `thal = 7` as a measured quantity without explanation?


# Part 2: Inspect the Continuous Predictors

Gaussian Naive Bayes models each continuous predictor within each class using a normal distribution.

For predictor $X_j$ and class $C_k$:

$$
P(X_j=x\mid C_k)
=
\frac{1}{\sqrt{2\pi\sigma_{jk}^2}}
\exp\left[
-\frac{(x-\mu_{jk})^2}{2\sigma_{jk}^2}
\right]
$$

The model estimates a separate mean and variance for each feature within each class.

The assumption does not require the overall feature distribution to be normal. The relevant question is whether the distribution within each class is reasonably compatible with the Gaussian approximation.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Why Gaussian Naive Bayes Uses Means and Variances

Gaussian Naive Bayes represents each continuous predictor within each class with a normal distribution. Its likelihood therefore depends on a class-specific mean and variance for every predictor. The model can still run when the shape is imperfect, but strong skew, outliers, or mixed subgroups can make that representation less credible.

### Deeper explanation

For class $k$ and feature $j$, Gaussian NB estimates $\mu_{jk}$ and $\sigma^2_{jk}$, then adds log densities across features. A value far from the class mean receives lower likelihood, scaled by class variance. Very small estimated variances can make scores numerically extreme, which motivates variance smoothing. The overall feature need not be Gaussian, but each class-conditional feature distribution is being approximated by one bell-shaped density.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 2: Visualize Continuous Features by Class

```text
Using heart and continuous_features:

1. create one separate histogram figure for each continuous predictor;
2. overlay the distributions for heartdisease = 0 and heartdisease = 1;
3. use the same bins for both classes within a feature;
4. label the axes, class legend, and title;
5. print a table of class-specific means and standard deviations;
6. do not fit a model.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 2


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 2


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for feature in continuous_features:
    feature_values = heart[feature].dropna()
    bins = np.histogram_bin_edges(feature_values, bins="auto")

    fig, ax = plt.subplots(figsize=(8, 5))
    for class_value in [0, 1]:
        class_values = heart.loc[
            heart[target_column] == class_value,
            feature,
        ].dropna()

        ax.hist(
            class_values,
            bins=bins,
            alpha=0.55,
            label=f"heartdisease = {class_value}",
        )

    ax.set_title(f"{feature} by Heart-Disease Class")
    ax.set_xlabel(feature)
    ax.set_ylabel("Frequency")
    ax.legend()
    plt.show()

class_summary = (
    heart.groupby(target_column)[continuous_features]
    .agg(["mean", "std"])
    .round(3)
)
display(class_summary)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

For each feature:

1. Is the shape roughly symmetric, skewed, multimodal, or irregular?
2. Do the two classes have different centers?
3. Do the classes have different spreads?
4. Are there extreme values?
5. Which feature appears least Gaussian?
6. Does a visible class difference guarantee useful classification?
7. Why should the model still be evaluated on held-out data?


# Part 3: Create One Locked Split and Fit Gaussian Naive Bayes

The same row split will be preserved for all comparisons in this tutorial.

A fair comparison requires:

- identical training rows;
- identical test rows;
- no tuning on the final test outcomes;
- the same target definition.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 3: Split and Fit the Continuous Model

```text
Using heart:

1. remove rows missing the target or any continuous feature and print how many
   rows are removed;
2. create one stratified 70/30 split of row indices with random_state = 18;
3. use only continuous_features for the required model;
4. fit sklearn.naive_bayes.GaussianNB on the training rows;
5. name the model gaussian_nb;
6. print training and test shapes and class proportions;
7. do not calculate final test metrics yet.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 3


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 3


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

required_level1_columns = continuous_features + [target_column]
level1_data = heart.dropna(subset=required_level1_columns).copy()

removed_rows = len(heart) - len(level1_data)
print(f"Rows removed for the Level 1 model: {removed_rows}")

train_index, test_index = train_test_split(
    level1_data.index,
    test_size=0.30,
    random_state=18,
    stratify=level1_data[target_column],
)

X_train_cont = level1_data.loc[train_index, continuous_features]
X_test_cont = level1_data.loc[test_index, continuous_features]
y_train = level1_data.loc[train_index, target_column].astype(int)
y_test = level1_data.loc[test_index, target_column].astype(int)

gaussian_nb = GaussianNB()
gaussian_nb.fit(X_train_cont, y_train)

print("Training shape:", X_train_cont.shape)
print("Test shape:", X_test_cont.shape)

print("\nTraining class proportions:")
display(y_train.value_counts(normalize=True).sort_index().to_frame("proportion"))

print("\nTest class proportions:")
display(y_test.value_counts(normalize=True).sort_index().to_frame("proportion"))


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

- Is the split stratified and reproducible?
- Are the same indices available for later comparisons?
- Did the model use only the five continuous features?
- Was the test set kept outside fitting?
- Did Gaussian Naive Bayes require feature scaling?
- Why or why not?
- What means and variances did the model estimate for each class?


# Part 4: Posterior Probabilities and Test Performance

For each test observation, Gaussian Naive Bayes produces one probability per class.

The predicted class is the class with the larger estimated posterior probability. For two classes, the default decision is equivalent to choosing class `1` when its posterior probability exceeds class `0`.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Probability, Class, and Error Are Different Objects

The posterior probability expresses model confidence under its assumptions. A decision rule converts that probability into a class, and the confusion matrix compares the class with the observed outcome. Accuracy summarizes all rows, while sensitivity and specificity ask different questions about the two actual outcome groups.

### Deeper explanation

The decision rule is $\hat C=\arg\max_k \hat P(C=k\mid x)$ under equal misclassification cost. With unequal costs, choose the action with smaller conditional expected loss, $a^*(x)=\arg\min_a\sum_k L(a,k)P(C=k\mid x)$. Thus a 0.50 cutoff is a cost assumption, not a law. Ranking, calibration, and decision utility should be evaluated separately.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 4: Evaluate the Gaussian Model

```text
Using gaussian_nb and the locked test set:

1. calculate the probability of heartdisease = 1;
2. calculate predicted classes;
3. display the first 10 test observations with actual class, predicted class,
   and predicted probability;
4. calculate accuracy, precision, sensitivity/recall, specificity, and ROC AUC;
5. display a metric table;
6. display a labeled confusion matrix;
7. display an ROC curve using probabilities;
8. use zero_division = 0 where appropriate;
9. name the probability vector heart_probability.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 4


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 4


In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

heart_probability = gaussian_nb.predict_proba(X_test_cont)[:, 1]
heart_prediction = gaussian_nb.predict(X_test_cont)

prediction_table = X_test_cont.copy()
prediction_table["ActualHeartDisease"] = y_test
prediction_table["PredictedHeartDisease"] = heart_prediction
prediction_table["ProbabilityHeartDisease"] = heart_probability

display(prediction_table.head(10))

heart_cm = confusion_matrix(y_test, heart_prediction)
tn, fp, fn, tp = heart_cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) else 0.0

heart_metrics = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Precision",
            "Sensitivity / Recall",
            "Specificity",
            "ROC AUC",
        ],
        "Value": [
            accuracy_score(y_test, heart_prediction),
            precision_score(y_test, heart_prediction, zero_division=0),
            recall_score(y_test, heart_prediction, zero_division=0),
            specificity,
            roc_auc_score(y_test, heart_probability),
        ],
    }
)
display(heart_metrics.style.format({"Value": "{:.3f}"}))

ConfusionMatrixDisplay(
    confusion_matrix=heart_cm,
    display_labels=["No heart disease", "Heart disease"],
).plot(values_format="d")
plt.title("Gaussian Naive Bayes Confusion Matrix")
plt.show()

fpr, tpr, thresholds = roc_curve(y_test, heart_probability)
heart_auc = roc_auc_score(y_test, heart_probability)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, label=f"Gaussian NB, AUC = {heart_auc:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", label="No-skill reference")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Gaussian Naive Bayes ROC Curve")
ax.legend()
plt.show()


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

1. How many false negatives occurred?
2. How many false positives occurred?
3. Does accuracy hide an imbalance between sensitivity and specificity?
4. Which observations have probabilities closest to 0.50?
5. Which incorrect predictions were made with high confidence?
6. Why might Naive Bayes probabilities be overconfident when predictors are dependent?
7. Does the model output establish that a person has or does not have heart disease?


# Part 5: Descriptive Error Audit by the Supplied Sex Code

The original dataset includes a binary `sex` code.

A subgroup audit can reveal whether aggregate performance hides different error patterns. It does not prove that a model is fair, and it does not explain why a difference exists.

The audit is descriptive because:

- the sample is limited;
- the code represents the dataset’s categories, not the full range of human identity;
- group sizes may differ;
- other variables may be distributed differently;
- this is not a clinical validation study.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 5: Compare Error Rates by `sex`

```text
Using the locked test_index, y_test, heart_prediction, and heart:

1. create an audit dataframe with sex, actual class, and predicted class;
2. for each observed sex code, calculate count, actual positive rate, accuracy,
   sensitivity, specificity, false positives, and false negatives;
3. use zero-safe calculations;
4. display one table;
5. do not label the model fair or unfair automatically.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 5


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 5


In [ ]:
audit_data = pd.DataFrame(
    {
        "sex": heart.loc[test_index, "sex"],
        "actual": y_test,
        "predicted": heart_prediction,
    },
    index=test_index,
)

audit_rows = []

for sex_code, group in audit_data.groupby("sex"):
    group_cm = confusion_matrix(
        group["actual"],
        group["predicted"],
        labels=[0, 1],
    )
    tn, fp, fn, tp = group_cm.ravel()

    audit_rows.append(
        {
            "sex": sex_code,
            "Count": len(group),
            "ActualPositiveRate": group["actual"].mean(),
            "Accuracy": accuracy_score(group["actual"], group["predicted"]),
            "Sensitivity": tp / (tp + fn) if (tp + fn) else float("nan"),
            "Specificity": tn / (tn + fp) if (tn + fp) else float("nan"),
            "FalsePositives": int(fp),
            "FalseNegatives": int(fn),
        }
    )

sex_error_audit = pd.DataFrame(audit_rows).sort_values("sex")
display(
    sex_error_audit.style.format(
        {
            "ActualPositiveRate": "{:.3f}",
            "Accuracy": "{:.3f}",
            "Sensitivity": "{:.3f}",
            "Specificity": "{:.3f}",
        }
    )
)


# AI for Social Good: A Prediction Is Not a Diagnosis

Health prediction may support social good when it:

- helps researchers identify patterns for further study;
- reports uncertainty and error rates;
- distinguishes screening from diagnosis;
- tests performance across relevant groups;
- is validated on appropriate populations;
- supports, rather than replaces, qualified clinical judgment.

It can cause harm when:

- a classroom model is presented as a medical tool;
- false negatives delay care;
- false positives produce anxiety or unnecessary procedures;
- category codes are interpreted carelessly;
- a small dataset is treated as representative of everyone;
- privacy risks are ignored;
- probability outputs are presented as certainty.

> **Social-good principle:** The more consequential the decision, the stronger the evidence, validation, transparency, and human oversight must be.


# Tutorial 6 Conclusion

You used Claude to:

1. inspect a processed heart-disease file;
2. separate continuous from discrete features;
3. examine Gaussian assumptions visually;
4. preserve one locked stratified split;
5. fit Gaussian Naive Bayes;
6. inspect posterior probabilities;
7. calculate several classification measures;
8. audit subgroup error descriptively.

The central lesson is that Naive Bayes gains simplicity from assumptions. Those assumptions must remain visible when interpreting the output.


# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Final Reflection

Answer briefly:

1. What is the difference between a prior and a posterior probability?
2. What does the conditional-independence assumption state?
3. Why did Level 1 use only the continuous predictors?
4. What does Gaussian Naive Bayes estimate for each feature and class?
5. Which metric is most sensitive to false negatives?
6. Why can a high accuracy still be inadequate?
7. What might cause overconfident probabilities?
8. What did the subgroup audit reveal?
9. What could it not establish?
10. Why is this notebook not a diagnostic tool?


# <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 — Optional Deep Dives

> **Challenge ahead:** Complete Level 1 first; this optional route adds a harder application of the same reading.

# Part 6: Compare Naive Bayes Representations

The original course notebook considered different Naive Bayes forms for different feature types.

This deep dive compares three models using the same row split:

1. **Continuous Gaussian NB:** the required Level 1 model;
2. **All-feature Gaussian NB:** a stress test that treats every numeric code as Gaussian;
3. **Discrete Bernoulli NB:** one-hot encodes the eight discrete predictors and models the resulting binary indicators.

The comparison does not create one combined clinical model. It asks how representation and assumptions change the result.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 6: Compare Three Naive Bayes Models

```text
Using only rows with complete values in all expected columns:

1. create one locked stratified 70/30 split with random_state = 18;
2. fit continuous-feature GaussianNB;
3. fit all-feature GaussianNB as an explicit assumption stress test;
4. fit a Pipeline containing OneHotEncoder(handle_unknown="ignore") and
   BernoulliNB for the discrete features;
5. evaluate each model on the same test rows using accuracy, sensitivity,
   specificity, and ROC AUC;
6. return one comparison dataframe;
7. do not declare one model universally best.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 6


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 6


In [ ]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

complete_data = heart.dropna(subset=expected_columns).copy()
all_features = continuous_features + discrete_features

level2_train_index, level2_test_index = train_test_split(
    complete_data.index,
    test_size=0.30,
    random_state=18,
    stratify=complete_data[target_column],
)

y_train_l2 = complete_data.loc[level2_train_index, target_column].astype(int)
y_test_l2 = complete_data.loc[level2_test_index, target_column].astype(int)

model_specs = {}

continuous_gaussian = GaussianNB()
continuous_gaussian.fit(
    complete_data.loc[level2_train_index, continuous_features],
    y_train_l2,
)
model_specs["Continuous Gaussian NB"] = (
    continuous_gaussian,
    continuous_features,
)

all_gaussian = GaussianNB()
all_gaussian.fit(
    complete_data.loc[level2_train_index, all_features],
    y_train_l2,
)
model_specs["All-Feature Gaussian NB"] = (
    all_gaussian,
    all_features,
)

discrete_bernoulli = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore"),
        ),
        (
            "bernoulli",
            BernoulliNB(alpha=1.0),
        ),
    ]
)
discrete_bernoulli.fit(
    complete_data.loc[level2_train_index, discrete_features],
    y_train_l2,
)
model_specs["Discrete Bernoulli NB"] = (
    discrete_bernoulli,
    discrete_features,
)

comparison_rows = []

for model_name, (model_object, feature_list) in model_specs.items():
    X_test_model = complete_data.loc[level2_test_index, feature_list]
    model_prediction = model_object.predict(X_test_model)
    model_probability = model_object.predict_proba(X_test_model)[:, 1]

    model_cm = confusion_matrix(y_test_l2, model_prediction)
    tn, fp, fn, tp = model_cm.ravel()

    comparison_rows.append(
        {
            "Model": model_name,
            "FeatureCount": len(feature_list),
            "Accuracy": accuracy_score(y_test_l2, model_prediction),
            "Sensitivity": tp / (tp + fn) if (tp + fn) else 0.0,
            "Specificity": tn / (tn + fp) if (tn + fp) else 0.0,
            "ROC_AUC": roc_auc_score(y_test_l2, model_probability),
        }
    )

naive_bayes_comparison = pd.DataFrame(comparison_rows)
display(
    naive_bayes_comparison.style.format(
        {
            "Accuracy": "{:.3f}",
            "Sensitivity": "{:.3f}",
            "Specificity": "{:.3f}",
            "ROC_AUC": "{:.3f}",
        }
    )
)


### Comparison Questions

1. Which representation has the highest sensitivity?
2. Which has the highest specificity?
3. Does the all-feature Gaussian stress test outperform the more defensible feature-specific models?
4. If it does, does that make the Gaussian treatment of category codes conceptually correct?
5. What information is lost by using only continuous features?
6. What information is lost by using only discrete features?
7. Why is a single test split not enough to establish a permanent ranking?


# Part 7: Audit the Conditional-Independence Assumption

Naive Bayes does not require predictors to be marginally independent. It assumes they are independent **within each class**.

A simple correlation check cannot prove independence, especially for nonlinear relationships or categorical features. It can, however, reveal obvious conflicts with the assumption.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: The ‘Naive’ Assumption Is Conditional

Naive Bayes assumes predictors are independent after the class is known, not independent in the full dataset. Correlation tables can reveal obvious dependence but cannot prove conditional independence. The audit is therefore evidence about model plausibility, not a pass–fail certificate.

### Deeper explanation

Conditional independence means $P(X_j\mid X_{-j},C)=P(X_j\mid C)$. Zero within-class correlation is weaker: it rules out only linear association. If two tests measure the same biological process, multiplying their likelihoods can double-count evidence and sharpen posteriors excessively. Naive Bayes may still classify well because the largest class score can be correct even when the normalized probability is too extreme.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 7: Find Strong Class-Conditional Correlations

```text
Using complete_data and continuous_features:

1. calculate the continuous-feature correlation matrix separately for
   heartdisease = 0 and heartdisease = 1;
2. list every unique feature pair and its correlation within each class;
3. calculate the absolute correlation;
4. display the five largest absolute correlations for each class;
5. do not claim that low correlation proves independence.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 7


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 7


In [ ]:
correlation_rows = []

for class_value in [0, 1]:
    class_correlation = (
        complete_data.loc[
            complete_data[target_column] == class_value,
            continuous_features,
        ]
        .corr()
    )

    for left_index, left_feature in enumerate(continuous_features):
        for right_feature in continuous_features[left_index + 1:]:
            correlation_value = class_correlation.loc[
                left_feature,
                right_feature,
            ]

            correlation_rows.append(
                {
                    "heartdisease": class_value,
                    "Feature1": left_feature,
                    "Feature2": right_feature,
                    "Correlation": correlation_value,
                    "AbsoluteCorrelation": abs(correlation_value),
                }
            )

conditional_correlations = pd.DataFrame(correlation_rows)

for class_value in [0, 1]:
    print(f"Five largest absolute correlations for class {class_value}:")
    display(
        conditional_correlations.loc[
            conditional_correlations["heartdisease"] == class_value
        ]
        .sort_values("AbsoluteCorrelation", ascending=False)
        .head(5)
        .reset_index(drop=True)
    )


### Independence Reflection

1. Which feature pairs have the strongest within-class correlations?
2. Are the strongest pairs the same in both classes?
3. Why does correlation measure only one kind of dependence?
4. How could dependence cause the model to count similar evidence more than once?
5. Why might ranking performance remain useful even when assumptions are imperfect?
6. Why should probability calibration be inspected separately?


# Part 8: Tune `var_smoothing` Without Touching the Test Set

`GaussianNB` uses `var_smoothing` to add a small amount to estimated variances. This can improve numerical stability.

The value must be compared using training data only. The untouched test set should not choose the smoothing value.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 8: Cross-Validate Smoothing Values

```text
Using only the Level 2 training rows and continuous_features:

1. compare var_smoothing values 1e-11, 1e-9, 1e-7, and 1e-5;
2. use StratifiedKFold with 5 folds, shuffle=True, and random_state = 1099;
3. calculate ROC AUC in each fold;
4. return mean and standard deviation of cross-validated ROC AUC;
5. sort from highest mean AUC to lowest;
6. do not evaluate the Level 2 test rows.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 8


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 8


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_l2_train_cont = complete_data.loc[
    level2_train_index,
    continuous_features,
]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=1099,
)

smoothing_rows = []

for smoothing_value in [1e-11, 1e-9, 1e-7, 1e-5]:
    candidate_model = GaussianNB(var_smoothing=smoothing_value)

    auc_scores = cross_val_score(
        candidate_model,
        X_l2_train_cont,
        y_train_l2,
        cv=cv,
        scoring="roc_auc",
    )

    smoothing_rows.append(
        {
            "var_smoothing": smoothing_value,
            "MeanCV_AUC": auc_scores.mean(),
            "SD_CV_AUC": auc_scores.std(ddof=1),
        }
    )

smoothing_comparison = (
    pd.DataFrame(smoothing_rows)
    .sort_values("MeanCV_AUC", ascending=False)
    .reset_index(drop=True)
)

display(
    smoothing_comparison.style.format(
        {
            "var_smoothing": "{:.0e}",
            "MeanCV_AUC": "{:.3f}",
            "SD_CV_AUC": "{:.3f}",
        }
    )
)


# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 Look Back

The deep dive should change how you describe the model.

Do not say:

> “Naive Bayes is accurate.”

Say something more specific:

> “On this processed Cleveland dataset and this evaluation design, the selected Naive Bayes representation produced the reported test performance. The result remains limited by the sample, feature coding, conditional-independence assumptions, and lack of clinical validation.”

That statement is less dramatic and more defensible.


# Sources and Course Resources

- James, Gareth, Daniela Witten, Trevor Hastie, and Robert Tibshirani. _An Introduction to Statistical Learning_, Section 4.4.4.
- Cleveland Heart Disease dataset source used by the original course notebook: [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Heart+Disease).
- Original course context: [10 Promising AI Applications in Health Care](https://hbr.org/2018/05/10-promising-ai-applications-in-health-care).
